# 02 — Image Server (Pollinations.ai + Flux fallback)

שרת FastAPI ליצירת תמונות אוואטר ותמונות רקע ל-thumbnails.

**Primary:** `Pollinations.ai` — חינמי, בלי מפתח, בלי GPU, מחזיר מיידית תמונה.
**Fallback:** `Flux.1-schnell` דרך `diffusers` — רץ מקומית על GPU אם Pollinations נופל.

Runtime מומלץ: **GPU (T4)**. אם אין GPU זה עדיין יעבוד — ה-fallback פשוט לא יהיה זמין.

## CELL 1 — התקנת ספריות

חבילות מינימליות בלבד. `diffusers` + `torch` **לא** מותקנים כאן — הם יותקנו רק אם ה-fallback ל-Flux ייפעל בפועל, כדי לחסוך זמן טעינה.

In [ ]:
# חבילות ליבה — FastAPI + HTTP client + עיבוד תמונה
!pip install -q fastapi uvicorn nest-asyncio httpx pillow aiofiles

## CELL 2 — חיבור Google Drive

תיקיית פלט יחידה לתמונות. אוואטרים נשמרים כ-`{job_id}.jpg`, thumbnails כ-`{job_id}_thumb.jpg`.

In [ ]:
# חיבור Drive — פותח חלון אימות בפעם הראשונה
from google.colab import drive
drive.mount('/content/drive')

import os

# תיקיית פלט יחידה ב-Drive
IMAGES_DIR = '/content/drive/MyDrive/viral_empire/outputs/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

print(f'IMAGES_DIR = {IMAGES_DIR}')

## CELL 3 — FastAPI app (אוואטרים + thumbnails) + בדיקת דגימה

שלושה endpoints:
- `POST /generate-avatar` — Pollinations בראש, Flux כגיבוי
- `POST /generate-thumbnail` — לוקח תמונת רקע ומוסיף כיתוב ויראלי ב-PIL
- `GET /health`

בסוף התא — קריאה אמיתית ל-Pollinations שמוודאת שהפייפליין עובד מקצה-לקצה.

In [ ]:
# ---------- בניית שרת FastAPI ----------
import asyncio
import io
import os
import threading
import time
import traceback
import urllib.parse
from typing import Optional

import httpx
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel, Field

# מאפשר event loop מקונן בתוך Colab
nest_asyncio.apply()

app = FastAPI(title='Viral Empire — Image Server', version='0.1.0')

# ה-pipeline של Flux נטען בעצלתיים (רק אם Pollinations נכשל)
_flux_pipe = None
_flux_lock = threading.Lock()

# ---------- Schemas ----------
class AvatarRequest(BaseModel):
    job_id: str = Field(..., min_length=1)
    prompt: str = Field(..., min_length=3)
    negative_prompt: str = Field(default='')
    width: int = Field(default=1080, ge=64, le=2048)
    height: int = Field(default=1350, ge=64, le=2048)
    seed: int = Field(default=42)

class ThumbnailRequest(BaseModel):
    job_id: str = Field(..., min_length=1)
    background_image_path: str = Field(..., description='Path to background image (Drive path OK)')
    text: str = Field(..., min_length=1, max_length=80)
    text_color: str = Field(default='#FFFFFF')
    stroke_color: str = Field(default='#000000')

# ---------- Pollinations primary ----------
async def _fetch_from_pollinations(req: AvatarRequest) -> bytes:
    # ה-API הוא GET-based: הפרומפט נכנס כחלק ב-path
    encoded_prompt = urllib.parse.quote(req.prompt, safe='')
    url = f'https://image.pollinations.ai/prompt/{encoded_prompt}'
    params = {
        'width': req.width,
        'height': req.height,
        'seed': req.seed,
        'model': 'flux',
        'nologo': 'true',
    }
    if req.negative_prompt:
        params['negative'] = req.negative_prompt

    # timeout נדיב — Pollinations לפעמים לוקח 20-40 שניות
    async with httpx.AsyncClient(timeout=90, follow_redirects=True) as http:
        resp = await http.get(url, params=params)
    resp.raise_for_status()
    if not resp.content or len(resp.content) < 1000:
        raise RuntimeError(f'pollinations returned empty/tiny response ({len(resp.content)} bytes)')
    return resp.content

# ---------- Flux fallback (lazy) ----------
def _ensure_flux_loaded():
    """מתקין ומטעין את Flux רק בפעם הראשונה שצריכים אותו."""
    global _flux_pipe
    if _flux_pipe is not None:
        return _flux_pipe
    with _flux_lock:
        if _flux_pipe is not None:
            return _flux_pipe
        print('[flux] installing diffusers + torch...')
        os.system('pip install -q diffusers transformers accelerate sentencepiece protobuf')
        import torch
        from diffusers import FluxPipeline

        if not torch.cuda.is_available():
            raise RuntimeError('flux fallback requires GPU — none available')

        print('[flux] loading FLUX.1-schnell (first run downloads ~24GB)')
        try:
            pipe = FluxPipeline.from_pretrained(
                'black-forest-labs/FLUX.1-schnell',
                torch_dtype=torch.bfloat16,
            )
            pipe.enable_model_cpu_offload()
            _flux_pipe = pipe
            print('[flux] loaded')
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise RuntimeError(f'flux OOM: {exc}') from exc
    return _flux_pipe

def _generate_with_flux(req: AvatarRequest) -> bytes:
    import torch
    pipe = _ensure_flux_loaded()
    generator = torch.Generator('cuda').manual_seed(int(req.seed))
    try:
        image = pipe(
            prompt=req.prompt,
            width=req.width,
            height=req.height,
            num_inference_steps=4,          # schnell — 4 צעדים מספיקים
            guidance_scale=0.0,             # schnell מכויל בלי CFG
            generator=generator,
        ).images[0]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=92)
    return buf.getvalue()

# ---------- /generate-avatar ----------
@app.post('/generate-avatar')
async def generate_avatar(req: AvatarRequest):
    out_path = os.path.join(IMAGES_DIR, f'{req.job_id}.jpg')
    provider = None
    image_bytes: Optional[bytes] = None
    errors = []

    # 1. ניסיון ראשון — Pollinations (ברירת מחדל)
    try:
        image_bytes = await _fetch_from_pollinations(req)
        provider = 'pollinations'
    except Exception as exc:
        print(f'[pollinations] failed: {exc}')
        errors.append(f'pollinations: {exc}')

    # 2. Fallback ל-Flux מקומי
    if image_bytes is None:
        try:
            image_bytes = await asyncio.to_thread(_generate_with_flux, req)
            provider = 'flux'
        except Exception as exc:
            traceback.print_exc()
            errors.append(f'flux: {exc}')
            raise HTTPException(503, f'all providers failed: {errors}') from exc

    # שמירה כ-JPEG (נורמליזציה דרך PIL כדי שלא נשמור PNG במקרה של Pollinations)
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img.save(out_path, format='JPEG', quality=92)

    return {
        'job_id': req.job_id,
        'image_url': out_path,                     # נתיב Drive
        'provider': provider,
        'size_bytes': os.path.getsize(out_path),
        'dimensions': [img.width, img.height],
    }

# ---------- /generate-thumbnail ----------
def _find_bold_font(size: int) -> ImageFont.FreeTypeFont:
    """מחפש פונט מודגש זמין במערכת (Colab מגיע עם DejaVu Sans Bold)."""
    candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        '/usr/share/fonts/truetype/noto/NotoSans-Bold.ttf',
    ]
    for path in candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

def _render_thumbnail(
    background_path: str,
    text: str,
    text_color: str,
    stroke_color: str,
    out_path: str,
):
    if not os.path.exists(background_path):
        raise FileNotFoundError(f'background not found: {background_path}')

    TARGET_W, TARGET_H = 1080, 1920

    # טעינה + resize עם שמירת יחס (cover)
    bg = Image.open(background_path).convert('RGB')
    src_ratio = bg.width / bg.height
    target_ratio = TARGET_W / TARGET_H
    if src_ratio > target_ratio:
        # רחב מדי — scale לפי גובה
        new_h = TARGET_H
        new_w = int(bg.width * (new_h / bg.height))
    else:
        new_w = TARGET_W
        new_h = int(bg.height * (new_w / bg.width))
    bg = bg.resize((new_w, new_h), Image.LANCZOS)
    # crop למרכז
    left = (new_w - TARGET_W) // 2
    top = (new_h - TARGET_H) // 2
    bg = bg.crop((left, top, left + TARGET_W, top + TARGET_H))

    draw = ImageDraw.Draw(bg)

    # בחירת גודל פונט: 140px ומצמצמים עד שהטקסט נכנס לרוחב
    font_size = 140
    MAX_W = int(TARGET_W * 0.86)
    font = _find_bold_font(font_size)
    while font_size > 80:
        bbox = draw.textbbox((0, 0), text, font=font, stroke_width=4)
        tw = bbox[2] - bbox[0]
        if tw <= MAX_W:
            break
        font_size -= 8
        font = _find_bold_font(font_size)

    bbox = draw.textbbox((0, 0), text, font=font, stroke_width=4)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]

    # מיקום: מרכז אופקי, 30% תחתונים
    x = (TARGET_W - tw) // 2 - bbox[0]
    y = int(TARGET_H * 0.70) - bbox[1]

    draw.text(
        (x, y),
        text,
        font=font,
        fill=text_color,
        stroke_width=4,
        stroke_fill=stroke_color,
    )

    bg.save(out_path, format='JPEG', quality=92)
    return {'width': TARGET_W, 'height': TARGET_H, 'font_size': font_size}

@app.post('/generate-thumbnail')
async def generate_thumbnail(req: ThumbnailRequest):
    out_path = os.path.join(IMAGES_DIR, f'{req.job_id}_thumb.jpg')
    try:
        meta = await asyncio.to_thread(
            _render_thumbnail,
            req.background_image_path,
            req.text,
            req.text_color,
            req.stroke_color,
            out_path,
        )
    except FileNotFoundError as exc:
        raise HTTPException(404, str(exc)) from exc
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(500, f'thumbnail failed: {exc}') from exc

    return {
        'job_id': req.job_id,
        'image_url': out_path,
        'size_bytes': os.path.getsize(out_path),
        **meta,
    }

# ---------- /health ----------
@app.get('/health')
def health():
    return {'status': 'ok', 'provider': 'pollinations'}

@app.get('/jobs/{job_id}/download')
def job_download(job_id: str):
    # מחפש גם avatar וגם thumbnail
    for suffix in ('.jpg', '_thumb.jpg'):
        p = os.path.join(IMAGES_DIR, f'{job_id}{suffix}')
        if os.path.exists(p):
            return FileResponse(p, media_type='image/jpeg', filename=os.path.basename(p))
    raise HTTPException(404, 'not found')

# ---------- הרצה ב-thread רקע ----------
PORT = 8000

def _run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# ממתינים שה-port נפתח
import socket
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except OSError:
        time.sleep(1)
print(f'FastAPI running on http://127.0.0.1:{PORT}')

# =============================================================
# בדיקת דגימה — מוודא שהפייפליין עובד מקצה-לקצה (Pollinations)
# =============================================================
print('\nRunning sample generation via Pollinations...')

async def _run_sample():
    sample_req = AvatarRequest(
        job_id='_sample',
        prompt='professional portrait of a young female influencer, soft studio lighting, ultra detailed, 4k',
        negative_prompt='low quality, blurry, deformed',
        width=768,
        height=1024,
        seed=1234,
    )
    return await generate_avatar(sample_req)

try:
    sample_result = asyncio.get_event_loop().run_until_complete(_run_sample())
    print(f'Sample OK: {sample_result}')
except Exception as exc:
    print(f'Sample failed: {exc}')

## CELL 4 — חשיפה דרך Cloudflare Tunnel

אותה טכניקה בדיוק כמו ב-notebook של TTS. מייצר URL `*.trycloudflare.com` ושומר ל-Drive כדי שה-backend המקומי יוכל למצוא אותו.

In [ ]:
# התקנת cloudflared והרמת tunnel
import os
import re
import subprocess
import time

if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Installing cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('cloudflared installed')

LOG_PATH = '/content/cloudflared_image.log'
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

proc = subprocess.Popen(
    [
        'cloudflared', 'tunnel', '--no-autoupdate',
        '--url', f'http://localhost:{PORT}',
        '--logfile', LOG_PATH,
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'cloudflared started (pid={proc.pid}), waiting for public URL...')

public_url = None
url_pattern = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
for _ in range(60):
    time.sleep(1)
    if not os.path.exists(LOG_PATH):
        continue
    with open(LOG_PATH, 'r') as f:
        log_text = f.read()
    m = url_pattern.search(log_text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError(f'Could not find tunnel URL. Check {LOG_PATH}')

# שמירה ל-Drive עבור ה-backend המקומי
URL_FILE = '/content/drive/MyDrive/viral_empire/image_server_url.txt'
with open(URL_FILE, 'w') as f:
    f.write(public_url + '\n')

print('=' * 60)
print(f'PUBLIC URL: {public_url}')
print(f'Saved to:   {URL_FILE}')
print('=' * 60)
print(f'Test it:    curl {public_url}/health')

## CELL 5 — לולאת Keep-Alive

שומר על ה-runtime פעיל. אל תסגור את הטאב.

In [ ]:
# Keep-alive — לא לסגור את הטאב
import time

print('Image server running. Keep this tab open.')
print(f'Public URL: {public_url}')

try:
    while True:
        time.sleep(30)
        print(f"Alive: {time.strftime('%H:%M:%S')}")
except KeyboardInterrupt:
    print('Stopped by user')